# 1.- S01 | Practica guiada: pensar con datos antes de programar

**Ciencia de Datos Aplicada a Negocio**  
Master Analitica de Negocios, UNIE | Edicion 2604  
Jaime Pineda | 04/05/2026

Este notebook acompana la primera sesion. La idea no es memorizar Python, sino recorrer el mismo viaje de las slides con pequenas celdas ejecutables:

```text
problema -> pregunta -> KPI -> datos -> evidencia -> decision
```

Al final tendras una primera version del **Project Canvas** del caso No-Show.

## 1.1.- Como usar este notebook

Trabajaremos en tres niveles:

1. **Marco mental:** CRISP-DM, EDA, DAMA y PECO.
2. **Python minimo:** variables, tipos, listas, diccionarios y condicionales.
3. **Dataset real:** primeras preguntas sobre citas medicas no asistidas.

Cuando una celda tenga codigo, ejecutala y pregunta: **que decision o validacion permite hacer esto?**

# 2.- Del problema al KPI

Caso guia del curso: una clinica pierde capacidad cuando un paciente no acude a su cita.

Primer paso: transformar el dolor de negocio en una metrica.

```text
tasa_no_show = citas_no_asistidas / citas_totales
```

Esta tasa es un **baseline**: la referencia contra la que compararemos cualquier mejora futura.

In [ ]:
total_appointments = 110_527
no_show_appointments = 22_319

no_show_rate = no_show_appointments / total_appointments

print("Citas totales:", total_appointments)
print("Pacientes que no asistieron:", no_show_appointments)
print("Tasa de no-show:", round(no_show_rate * 100, 1), "%")

## 2.1.- Lagging vs leading

- `tasa_no_show` es un **KPI lagging**: mide un resultado que ya ocurrio.
- `waiting_days` o `% SMS enviados a tiempo` pueden ser **KPIs leading**: palancas que la organizacion puede intentar modificar.

Las decisiones se toman sobre leading. Los lagging nos dicen si funcionaron.

In [ ]:
kpis = {
    "lagging": "no_show_rate",
    "leading_1": "waiting_days",
    "leading_2": "sms_sent_on_time",
}

print(kpis)

## 2.2.- Microejercicio

La clinica estima estos valores para una semana:

```python
total_appointments = 500
missed_appointments = 95
cost_per_missed_appointment = 50
```

Tareas:

1. Calcula la tasa de no-show.
2. Calcula la perdida estimada.
3. Imprime una frase de negocio con ambos resultados.

In [ ]:
# Respuesta orientativa: completa esta celda durante la clase.
total_appointments = 500
missed_appointments = 95
cost_per_missed_appointment = 50

no_show_rate_week = missed_appointments / total_appointments
estimated_loss = missed_appointments * cost_per_missed_appointment

print("Tasa semanal de no-show:", round(no_show_rate_week * 100, 1), "%")
print("Perdida estimada semanal:", estimated_loss, "euros")
print(
    "Lectura de negocio: si se pierden",
    missed_appointments,
    "citas de",
    total_appointments,
    "la clinica deja de aprovechar aproximadamente",
    estimated_loss,
    "euros de capacidad esa semana."
)

# 3.- Python minimo viable

Python aparece aqui como herramienta para hacer reproducible el razonamiento.

No buscamos programar como desarrolladores. Buscamos **nombrar**, **calcular**, **aplicar reglas** y **validar resultados**.

In [ ]:
no_show_rate = 0.202
appointment = {
    "patient_id": 1001,
    "age": 34,
    "waiting_days": 18,
    "sms_received": False,
}

print("Tasa guardada:", no_show_rate)
print("Cita:", appointment)
print("Edad:", appointment["age"])

## 3.1.- Tipos: que clase de dato tengo?

Los tipos ayudan a evitar errores silenciosos. Una edad, una tasa y una etiqueta no se tratan igual.

In [ ]:
print("age:", appointment["age"], "->", type(appointment["age"]))
print("waiting_days:", appointment["waiting_days"], "->", type(appointment["waiting_days"]))
print("sms_received:", appointment["sms_received"], "->", type(appointment["sms_received"]))
print("no_show_rate:", no_show_rate, "->", type(no_show_rate))

## 3.2.- Listas: pequenas colecciones

Una lista nos permite representar varios valores relacionados. Aqui `True` significa que el paciente falto a la cita.

In [ ]:
missed_history = [False, True, False, False, True]

missed_count = sum(missed_history)
appointment_count = len(missed_history)
history_no_show_rate = missed_count / appointment_count

print("Citas en el historial:", appointment_count)
print("Inasistencias:", missed_count)
print("Tasa historica:", round(history_no_show_rate * 100, 1), "%")

## 3.3.- Condicionales: de intuicion a regla

Una regla no es un modelo. Es una primera formalizacion de una intuicion de negocio.

```text
waiting_days > 14 y sin SMS -> riesgo alto
waiting_days > 7           -> riesgo medio
resto                      -> riesgo bajo
```

In [ ]:
waiting_days = appointment["waiting_days"]
sms_received = appointment["sms_received"]

if waiting_days > 14 and not sms_received:
    risk_level = "alto"
elif waiting_days > 7:
    risk_level = "medio"
else:
    risk_level = "bajo"

print("Nivel de riesgo:", risk_level)

## 3.4.- Microejercicio

Cambia los valores de `waiting_days` y `sms_received` para comprobar que la regla cambia de salida.

Pregunta clave: **que decision operativa podria activar una etiqueta de riesgo alto?**

In [ ]:
# Respuesta orientativa: prueba distintos escenarios.
waiting_days = 3
sms_received = True

if waiting_days > 14 and not sms_received:
    risk_level = "alto"
elif waiting_days > 7:
    risk_level = "medio"
else:
    risk_level = "bajo"

print("Escenario de prueba")
print("waiting_days:", waiting_days)
print("sms_received:", sms_received)
print("Nivel de riesgo:", risk_level)
print("Decision candidata: mantener gestion habitual; no parece necesaria una intervencion prioritaria.")

# 4.- Cargar el dataset real

Entramos en CRISP-DM fase 2: **Data Understanding**.

Usaremos el CSV original de Kaggle: `KaggleV2-May-2016.csv`. En Colab, si no esta en el repo, subelo a `/content`.

## 4.1.- Opcional Google Colab: preparar el CSV

Esta subseccion solo hace falta si estamos trabajando en **Google Colab** y el dataset no viene incluido con el notebook.

El archivo que necesitamos se llama `KaggleV2-May-2016.csv`. En Colab puede cargarse de dos formas sencillas:

1. Subirlo desde el panel lateral de archivos.
2. Ejecutar la siguiente celda opcional y seleccionarlo desde el ordenador.

Si estas en VS Code, Jupyter local o el CSV ya esta en el repositorio, puedes saltar directamente a la celda de carga principal.

In [ ]:
from pathlib import Path

csv_name = "KaggleV2-May-2016.csv"
colab_path = Path("/content") / csv_name

if Path("/content").exists() and not colab_path.exists():
    try:
        from google.colab import files

        print("Colab detectado. Selecciona KaggleV2-May-2016.csv.")
        uploaded_files = files.upload()
        print("Archivos subidos:", list(uploaded_files.keys()))
    except ModuleNotFoundError:
        print("No estas en Google Colab. Puedes saltar esta celda.")
elif colab_path.exists():
    print("CSV detectado en /content. Puedes continuar.")
else:
    print("Entorno local detectado. Puedes saltar esta celda opcional.")

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 50)

csv_name = "KaggleV2-May-2016.csv"
candidate_paths = [
    Path("/content") / csv_name,
    Path(csv_name),
    Path("Recursos/Datasets/noshowappointments") / csv_name,
    Path("../Recursos/Datasets/noshowappointments") / csv_name,
    Path("../../Recursos/Datasets/noshowappointments") / csv_name,
    Path("../../../Recursos/Datasets/noshowappointments") / csv_name,
    Path("../../../../Recursos/Datasets/noshowappointments") / csv_name,
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "No encuentro KaggleV2-May-2016.csv. "
        "En Colab, subelo a /content. En local, revisa la ruta del repo."
    )

df = pd.read_csv(data_path)

print(f"Archivo cargado: {data_path}")
display(df.head(3))

## 4.2.- Control de orientacion

Antes de analizar, comprobamos que estamos usando el dataset esperado.

Recordatorio critico: `No-show = "Yes"` significa que el paciente **no asistio**.

In [ ]:
expected_total_appointments = 110_527
expected_no_show_appointments = 22_319

orientation_check = pd.DataFrame(
    {
        "expected": [expected_total_appointments, expected_no_show_appointments],
        "observed": [len(df), df["No-show"].eq("Yes").sum()],
    },
    index=["citas_totales", "no_shows"],
)
orientation_check["match"] = orientation_check["expected"].eq(orientation_check["observed"])

actual_no_show_rate = df["No-show"].eq("Yes").mean()

display(orientation_check)
print(f"Baseline observado: {actual_no_show_rate:.1%}")

if not orientation_check["match"].all():
    print("Aviso: el CSV cargado no coincide con los valores esperados del curso.")

# 5.- EDA inicial: 5 preguntas canonicas

EDA no es mirar datos al azar. En esta asignatura usaremos siempre estas cinco preguntas:

1. **Forma:** que tamano y estructura tienen los datos?
2. **Unidad:** que representa una fila?
3. **Objetivo:** como se reparte la variable que importa?
4. **Anomalias:** hay valores imposibles o raros?
5. **Variables nuevas:** que necesitamos construir?

A partir de aqui usamos una caja minima de atajos Pandas: `head()`, `info()`, `describe()`, `value_counts()`, `isna()` y `nunique()`.

## 5.1.- EDA pregunta 1: forma

Pregunta: **que tamano y estructura tiene el dataset?**

In [ ]:
df.info()

display(df.dtypes.value_counts().to_frame(name="columns"))

## 5.2.- EDA pregunta 2: unidad de observacion

Pregunta: **que representa una fila?**

Pista: no es exactamente un paciente; un paciente puede tener mas de una cita.

In [ ]:
display(df.head())

display(
    df[["PatientId", "AppointmentID"]]
    .nunique()
    .to_frame(name="unique_values")
)

### Lectura del resultado

Una fila representa una **cita**, no un paciente. Por eso hay menos pacientes unicos que citas unicas.

Esta distincion importa: si analizamos como si cada fila fuera un paciente, podriamos contar mal el fenomeno.

## 5.3.- EDA pregunta 3: distribucion del objetivo

Pregunta: **como se reparte la variable que nos importa?**

Aqui hay un riesgo DAMA de **consistencia**: el nombre `No-show` puede inducir a interpretar al reves los valores.

In [ ]:
target_order = ["No", "Yes"]

target_summary = (
    df["No-show"]
    .value_counts(dropna=False)
    .reindex(target_order)
    .to_frame(name="appointments")
)
target_summary["percent"] = (
    df["No-show"]
    .value_counts(normalize=True, dropna=False)
    .reindex(target_order)
    .mul(100)
    .round(1)
)

baseline_no_show_rate = target_summary.loc["Yes", "percent"]

display(target_summary)

In [ ]:
target_plot = target_summary.rename(
    index={"No": "Asistio\n(No)", "Yes": "No asistio\n(Yes)"}
)

fig, ax = plt.subplots(figsize=(10, 6))
target_plot["appointments"].plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
ax.set_title("Distribucion de la variable objetivo No-show")
ax.set_xlabel("Interpretacion del valor original")
ax.set_ylabel("Numero de citas")
ax.tick_params(axis="x", rotation=0)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5.4.- EDA pregunta 4: anomalias

Pregunta: **hay valores imposibles, atipicos o inconsistentes?**

Empezamos por algo muy simple: la edad no puede ser negativa.

In [ ]:
age_min = df["Age"].min()
invalid_age_rows = df.query("Age < 0")[["PatientId", "AppointmentID", "Age", "No-show"]]
invalid_age_count = len(invalid_age_rows)

display(df["Age"].describe().to_frame().T)
display(invalid_age_rows)

### Etiqueta DAMA

Una edad negativa no es solo un numero raro. Es un problema de **validez**: el valor no es posible en el proceso real.

## 5.5.- DAMA rapido: nulos, consistencia y unicidad

DAMA ayuda a poner nombre a los problemas: completitud, validez, consistencia, exactitud y unicidad.

In [ ]:
missing_summary = df.isna().sum().to_frame(name="missing_values")
missing_summary["missing_percent"] = df.isna().mean().mul(100).round(2)

identity_summary = (
    df[["PatientId", "AppointmentID"]]
    .nunique()
    .to_frame(name="unique_values")
)
identity_summary["repeated_values"] = len(df) - identity_summary["unique_values"]

column_name_check = pd.Series(
    {
        "Hipertension": "Hipertension" in df.columns,
        "Handcap": "Handcap" in df.columns,
    },
    name="exists",
).to_frame()

display(missing_summary)
display(identity_summary)
display(column_name_check)

## 5.6.- EDA pregunta 5: variables nuevas

Pregunta: **que variables hay que construir?**

El tiempo de espera no viene como columna directa. Lo derivamos a partir de `ScheduledDay` y `AppointmentDay`. Esto anticipa feature engineering, que se trabajara con mas detalle en S03.

In [ ]:
df_demo = df[df["Age"] >= 0].copy()

df_demo["scheduled_day"] = pd.to_datetime(df_demo["ScheduledDay"], errors="coerce")
df_demo["appointment_day"] = pd.to_datetime(df_demo["AppointmentDay"], errors="coerce")

df_demo["waiting_days"] = (
    df_demo["appointment_day"].dt.normalize()
    - df_demo["scheduled_day"].dt.normalize()
).dt.days

waiting_days_check = (
    df_demo["waiting_days"]
    .lt(0)
    .value_counts()
    .rename(index={False: "espera_no_negativa", True: "espera_negativa"})
    .to_frame(name="appointments")
)

display(df_demo["waiting_days"].describe().to_frame().T)
display(waiting_days_check)

### Nota metodologica

Normalizamos las fechas antes de restar para trabajar con dias calendario. Si restamos fecha y hora completas, muchas citas del mismo dia parecen tener espera negativa por el timestamp.

Esto no es una limpieza definitiva: es una decision pedagogica para S01. En S02-S03 volveremos con mas control tecnico.

## 5.7.- Miniresumen EDA + DAMA

| Pregunta EDA | Resultado observado | Lectura DAMA |
|---|---|---|
| Forma | 110,527 filas y 14 columnas | Punto de partida tecnico |
| Unidad | Una fila = una cita | Riesgo de interpretar como paciente |
| Objetivo | `Yes` = no asistio; baseline 20.2% | Consistencia |
| Anomalias | 1 edad negativa | Validez |
| Variable nueva | `waiting_days` derivada de fechas | Feature engineering; revisar negativos |

Este resumen es el puente hacia PECO: ya no tenemos solo intuiciones, tenemos una primera evidencia organizada.

# 6.- PECO: de intuicion a hipotesis comprobable

PECO ordena una pregunta observacional:

- **Poblacion:** a quien aplica.
- **Exposicion/factor:** variable observada antes del resultado.
- **Comparacion:** contra que referencia.
- **Outcome:** diferencia esperada en el KPI.

PECO ayuda a formular preguntas comprobables. **No prueba causalidad por si solo.**

## 6.1.- Hipotesis PECO del caso

Hipotesis:

> En pacientes de la clinica brasilena (P), tener espera **>14 dias** (E), respecto a espera **<=14 dias** (C), se asocia con una **tasa de no-show mayor** (O).

Ahora calculamos una primera evidencia descriptiva.

In [ ]:
analysis_df = df_demo[df_demo["waiting_days"] >= 0].copy()
analysis_df["no_show_flag"] = analysis_df["No-show"].eq("Yes").astype(int)
analysis_df["waiting_group"] = analysis_df["waiting_days"].gt(14).map(
    {True: ">14 dias", False: "<=14 dias"}
)

peco_summary = analysis_df.groupby("waiting_group").agg(
    appointments=("AppointmentID", "count"),
    no_show_rate=("no_show_flag", "mean"),
)
peco_summary["no_show_rate"] = (peco_summary["no_show_rate"] * 100).round(1)

display(peco_summary.loc[["<=14 dias", ">14 dias"]])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
peco_summary.loc[["<=14 dias", ">14 dias"], "no_show_rate"].plot(
    kind="bar",
    ax=ax,
    color=["#4C78A8", "#F58518"],
)
ax.set_title("Tasa de no-show por tiempo de espera")
ax.set_xlabel("Grupo de espera")
ax.set_ylabel("Tasa de no-show (%)")
ax.tick_params(axis="x", rotation=0)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6.2.- Interpretacion cauta

Una frase correcta seria:

> En este dataset, las citas con espera superior a 14 dias muestran una tasa de no-show mayor que las citas con espera de 14 dias o menos.

Una frase demasiado fuerte seria:

> Esperar mas de 14 dias causa que el paciente no acuda.

Hoy hemos observado una asociacion descriptiva. Para hablar de causalidad necesitariamos otro diseno y mas controles.

## 6.3.- Microejercicio PECO

Reescribe esta intuicion en formato PECO:

> Los jovenes faltan mas a las citas.

Completa:

```text
Poblacion:
Exposicion/factor:
Comparacion:
Outcome esperado:
```

Despues, identifica que columna del dataset permitiria empezar a comprobarla.

### Respuesta orientativa

```text
Poblacion: citas medicas registradas en la clinica.
Exposicion/factor: pacientes jovenes, por ejemplo edad < 30 anos.
Comparacion: pacientes de 30 anos o mas.
Outcome esperado: mayor tasa de no-show en el grupo joven.
```

Columna para empezar a comprobarlo: `Age`, combinada con `No-show` para calcular la tasa de inasistencia por grupo de edad.

Frase prudente: en este dataset podemos comprobar si las citas de pacientes jovenes muestran una tasa de no-show mayor, pero no concluir causalidad solo con esta comparacion descriptiva.

# 7.- IA como copiloto tecnico

Puedes usar IA para pedir sintaxis, explicar errores o generar una primera version de codigo. No puedes delegar el criterio analitico.

Un buen prompt tecnico incluye:

1. Contexto de negocio.
2. Nombre de la tabla.
3. Columnas criticas.
4. Resultado esperado.
5. Validacion pedida.
6. Restriccion: no inventar columnas.

## 7.1.- Prompt aceptable para este caso

```text
Tengo un DataFrame llamado df con citas medicas.
La columna "No-show" tiene "Yes" cuando el paciente falto y "No" cuando asistio.
Quiero calcular la tasa total de no-show y ver tambien los conteos absolutos.
Genera codigo Pandas simple, explica que hace cada linea y anade una comprobacion
para no interpretar al reves la variable objetivo.
No inventes columnas que no te he descrito.
```

# 8.- Project Canvas v1

El canvas guarda el razonamiento para refinarlo durante el curso. Plantilla: [`../S01_2604_project_canvas.md`](../S01_2604_project_canvas.md).

Completa una primera version con lo que ya sabemos:

| Caja | Borrador S01 |
|---|---|
| Problema | Citas medicas no asistidas generan coste e ineficiencia. |
| KPI | Tasa de no-show, baseline 20.2%. |
| Datos | Kaggle, 110,527 citas, 14 columnas, una fila = una cita. |
| DAMA | Consistencia en `No-show`; validez en edad negativa; exactitud pendiente de negocio. |
| PECO | Espera >14 dias vs <=14 dias y tasa de no-show. |
| Decision candidata | Revisar gestion de citas con mucha espera sin penalizar acceso. |

# 9.- Cierre

Narrativa que deberias poder explicar al salir:

> Tenemos un problema de inasistencia. Estamos en Business Understanding y primera entrada a Data Understanding. Medimos el problema con una tasa baseline de no-show del 20.2%. Hicimos un EDA inicial con 5 preguntas, etiquetamos riesgos DAMA y formulamos una hipotesis PECO. Aun no hay modelo: hay una pregunta mejor formulada, una metrica y un primer canvas.

S02-S04 profundizaran en Pandas, limpieza y visualizacion. S05-S10 entraran en modelos y evaluacion contra baseline.